<a href="https://colab.research.google.com/github/wundertater/FerroCalc/blob/master/pinns_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q deepxde

In [ ]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

import time
import random
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import deepxde as dde

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
dde.config.set_random_seed(SEED)

dde.config.set_default_float("float32")

In [ ]:
t = torch.linspace(0, 5, 500)

Аналитическое решение

In [ ]:
def analytical_solution(t):
    """
    Exact solution of:
        x' + 2x + y = 0
        y' + x + 2y = 0

    x(0) = 1
    y(0) = 0
    """
    x = 0.5 * torch.exp(-t) + 0.5 * torch.exp(-3.0 * t)
    y = -0.5 * torch.exp(-t) + 0.5 * torch.exp(-3.0 * t)

    return torch.cat([x, y], dim=1)

/tmp/ipykernel_456/3803342663.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x_analitical = 0.5 * np.exp(-t) + 0.5 * np.exp(-3 * t)
/tmp/ipykernel_456/3803342663.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  y_analitical = - 0.5 * np.exp(-t) + 0.5 * np.exp(-3 * t)


In [ ]:
#simple
class SimplePinn(nn.Module):
    def __init__(self):
        super().__init__()

        activation = nn.Tanh()

        self.first_layer = nn.Sequential(nn.Linear(1, 64),
                                         activation)
        self.hidden_layer1 = nn.Sequential(nn.Linear(64, 64),
                                         activation)
        self.hidden_layer2 = nn.Sequential(nn.Linear(64, 64),
                                         activation)
        self.last_layer = nn.Sequential(nn.Linear(64, 2))

    def forward(self, x):
        x = self.first_layer(x)
        x = self.hidden_layer1(x)
        x = self.hidden_layer2(x)
        x = self.last_layer(x)

        return x

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width, activation):
        super().__init__()
        self.fc1 = nn.Linear(width, width)
        self.activation = activation

    def forward(self, x):
        return self.activation(x + self.activation(self.fc1(x)))


class PINN(nn.Module):
    def __init__(self, dim_hidden, num_layers=4):
        super().__init__()

        activation1 = nn.Tanh()

        self.hidden_layer = nn.Sequential(*[ResidualBlock(dim_hidden, activation1)
                                                for _ in range(num_layers)])

        self.first_layer = nn.Sequential(nn.Linear(2, dim_hidden),
                                         activation1)

        self.last_layer = nn.Sequential(nn.Linear(dim_hidden, 1))

    def forward(self, x):
        out = torch.cat([torch.sin(x), torch.cos(x)], dim=1)

        out = self.first_layer(out)
        out = self.hidden_layer(out)
        out = self.last_layer(out)

        return out

In [ ]:
def loss_fn(model, t):
    output = model(t)
    x = output[:, 0:1]
    y = output[:, 1:2]

    dx_dt = torch.autograd.grad(
    outputs=x,
    inputs=t,
    grad_outputs=torch.ones_like(x),
    create_graph=True)[0]

    dy_dt = torch.autograd.grad(
    outputs=y,
    inputs=t,
    grad_outputs=torch.ones_like(y),
    create_graph=True)[0]

    res_x = dx_dt + 2*x + y
    res_y = dy_dt + x + 2*y

    init_loss_x = torch.square(x[0] - 1)
    init_loss_y = torch.square(y[0])

    loss = torch.mean(torch.square(res_x)) + torch.mean(torch.square(res_y)) + init_loss_x + init_loss_y
    return loss

In [ ]:
def train(model, t, epochs, optimizer):
    for epoch in range(1, epochs+1):
        loss = loss_fn(model, t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f'Epoch: {epoch}, loss: {loss.item()}')

In [ ]:
pinn = SimplePinn()
optimizer = torch.optim.Adam(pinn.parameters(), lr=0.001)

In [ ]:
t_points = torch.tensor(t[:, None])
t_points.requires_grad_(True);

/tmp/ipykernel_456/3543813239.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_points = torch.tensor(t[:, None])


In [ ]:
train(pinn, t_points, 5000, optimizer)

Epoch: 0, loss: 0.9968921542167664
Epoch: 500, loss: 0.0049037085846066475
Epoch: 1000, loss: 1.1776531209761743e-05
Epoch: 1500, loss: 1.1140135939058382e-05
Epoch: 2000, loss: 7.294893748621689e-06
Epoch: 2500, loss: 4.76314426123281e-06
Epoch: 3000, loss: 3.0038247587071965e-06
Epoch: 3500, loss: 1.874145141300687e-06
Epoch: 4000, loss: 1.2437845953172655e-06
Epoch: 4500, loss: 1.6667171394146862e-06


In [ ]:
def calculate_metrics(y_true, y_pred):
    error = y_pred - y_true

    # Абсолютная ошибка
    abs_error = torch.abs(error)

    # MAE
    mae = torch.mean(abs_error)

    # RMSE
    rmse = torch.sqrt(torch.mean(error ** 2))

    # Максимальная абсолютная ошибка
    max_error = torch.max(abs_error)

    # Relative L2 error
    relative_l2 = (
        torch.linalg.norm(error)
        / torch.linalg.norm(y_true)
    )

    # Relative L2 error отдельно для x и y
    relative_l2_x = (
        torch.linalg.norm(error[:, 0])
        / torch.linalg.norm(y_true[:, 0])
    )

    relative_l2_y = (
        torch.linalg.norm(error[:, 1])
        / torch.linalg.norm(y_true[:, 1])
    )

    # R²
    ss_res = torch.sum(error ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true, dim=0)) ** 2)

    r2 = 1.0 - ss_res / ss_tot

    # Ошибка в начальном условии
    initial_error = torch.abs(error[0])

    metrics = {
        "Relative L2": relative_l2.item(),
        "RMSE": rmse.item(),
        "MAE": mae.item(),
        "Max Absolute Error": max_error.item(),
        "Relative L2 x": relative_l2_x.item(),
        "Relative L2 y": relative_l2_y.item(),
        "R2": r2.item(),
        "Initial Error x": initial_error[0].item(),
        "Initial Error y": initial_error[1].item(),
    }

    return metrics

In [ ]:
def physics_metrics(model, t):
    model.eval()

    t = t.clone().detach().requires_grad_(True)

    output = model(t)

    x = output[:, 0:1]
    y = output[:, 1:2]

    dx_dt = torch.autograd.grad(
        outputs=x,
        inputs=t,
        grad_outputs=torch.ones_like(x),
        create_graph=False
    )[0]

    dy_dt = torch.autograd.grad(
        outputs=y,
        inputs=t,
        grad_outputs=torch.ones_like(y),
        create_graph=False
    )[0]

    residual_x = dx_dt + 2.0 * x + y
    residual_y = dy_dt + x + 2.0 * y

    physics_mse_x = torch.mean(residual_x ** 2)
    physics_mse_y = torch.mean(residual_y ** 2)

    physics_mse = physics_mse_x + physics_mse_y

    physics_rmse = torch.sqrt(physics_mse)

    max_residual = torch.max(
        torch.cat([
            torch.abs(residual_x),
            torch.abs(residual_y)
        ])
    )

    return {
        "Physics MSE": physics_mse.item(),
        "Physics RMSE": physics_rmse.item(),
        "Physics MSE x": physics_mse_x.item(),
        "Physics MSE y": physics_mse_y.item(),
        "Max Physics Residual": max_residual.item(),
    }

In [ ]:
t_test = torch.linspace(
    0.0,
    5.0,
    1000,
    dtype=torch.float32
).reshape(-1, 1)

physics = physics_metrics(pinn, t_test)

print("\n" + "=" * 50)
print("PHYSICS RESIDUAL")
print("=" * 50)

for name, value in physics.items():
    print(f"{name:25s}: {value:.10e}")

pinn.eval()

with torch.no_grad():
    pinn_prediction = pinn(t_test)

    exact_prediction = analytical_solution(t_test)

metrics = calculate_metrics(
    exact_prediction,
    pinn_prediction
)

print("=" * 50)
print("PINN vs ANALYTICAL SOLUTION")
print("=" * 50)

for name, value in metrics.items():
    print(f"{name:25s}: {value:.10e}")